# Experiment 4: Bag-of-Words and N-grams

- **Bag-of-Words (BoW):** each document = word counts, order ignored. Stack them -> Document-Term Matrix.
- **N-grams:** N words in a row (bigram = 2, trigram = 3). Keep local word order.
- **Zipf's law:** frequency ~ 1 / rank, so a log-log plot is a straight line with slope about -1.

In [ ]:
# !pip install nltk scikit-learn pandas matplotlib

In [ ]:
import re
import nltk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from nltk import FreqDist
from nltk.util import ngrams
from nltk.corpus import stopwords, gutenberg
from sklearn.feature_extraction.text import CountVectorizer

for r in ['punkt', 'punkt_tab', 'stopwords', 'gutenberg']:
    nltk.download(r, quiet=True)
stop = set(stopwords.words('english'))

In [ ]:
corpus = [
    'Natural language processing enables computers to understand human language.',
    'Machine learning models learn patterns from large text data.',
    'Deep learning has improved natural language processing tasks.',
    'Text classification is a common natural language processing application.',
    'The bag of words model represents text data as word frequency vectors.',
]

def clean(text):
    words = re.sub(r'[^a-z\s]', ' ', text.lower()).split()
    return [w for w in words if w not in stop]

docs = [clean(d) for d in corpus]
docs[0]

## Bag-of-Words with scikit-learn

In [ ]:
vec = CountVectorizer()
X = vec.fit_transform([' '.join(d) for d in docs])   # sparse matrix

dtm = pd.DataFrame(X.toarray(), columns=vec.get_feature_names_out())
print('Shape (documents, vocabulary):', dtm.shape)
print(f'Sparsity: {(dtm.values == 0).mean():.0%} of cells are zero')
dtm

## Unigram, bigram, trigram frequencies

In [ ]:
words = [w for d in docs for w in d]
bigrams = [g for d in docs for g in ngrams(d, 2)]    # built per document
trigrams = [g for d in docs for g in ngrams(d, 3)]

print('Top unigrams:', FreqDist(words).most_common(5))
print('Top bigrams :', FreqDist(bigrams).most_common(5))
print('Top trigrams:', FreqDist(trigrams).most_common(3))

top = FreqDist(words).most_common(10)
plt.barh([w for w, _ in top][::-1], [c for _, c in top][::-1])
plt.title('Top 10 words'); plt.xlabel('Count'); plt.show()

## Bigger N -> more features, sparser matrix

In [ ]:
for n in (1, 2, 3):
    X = CountVectorizer(ngram_range=(n, n)).fit_transform([' '.join(d) for d in docs]).toarray()
    print(f'{n}-gram: vocabulary={X.shape[1]:3}  sparsity={(X == 0).mean():.0%}')

## Zipf's law on Jane Austen's *Emma*

In [ ]:
emma = [w.lower() for w in gutenberg.words('austen-emma.txt') if w.isalpha()]
fd = FreqDist(emma)
print('Total words:', fd.N(), '| Vocabulary:', fd.B(), '| Words seen once:', len(fd.hapaxes()))

freqs = np.array(sorted(fd.values(), reverse=True))
ranks = np.arange(1, len(freqs) + 1)
slope = np.polyfit(np.log10(ranks[:1000]), np.log10(freqs[:1000]), 1)[0]
print(f'Log-log slope (top 1000 words): {slope:.2f}  (Zipf predicts about -1)')

plt.loglog(ranks, freqs)
plt.xlabel('Rank'); plt.ylabel('Frequency'); plt.title("Zipf's law in Emma"); plt.show()